# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [14]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm

import joblib

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


### класс для извлечения признаков из дф

In [21]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    # preprocessing
    def transform(self, X):
        X_transformed = X.copy()
        X_transformed['timestamp'] = pd.to_datetime(X_transformed['timestamp'])

        X_transformed['hour'] = X_transformed['timestamp'].dt.hour
        X_transformed['dayofweek'] = X_transformed['timestamp'].dt.weekday

        X_transformed.drop(columns=['timestamp'], inplace=True)
        
        return X_transformed

### кодируем категориальные признаки

In [22]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.categorical_cols = []

    def fit(self, X, y=None):
        self.categorical_cols = [
            col for col in X.columns 
            if X[col].dtype == 'object' and col != self.target_col
        ]

        if self.categorical_cols:
            self.encoder.fit(X[self.categorical_cols])
            
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # определение целевой колонки, отделяем таргет
        y = None
        if self.target_col in X_transformed.columns:
            y = X_transformed[self.target_col]
            X_transformed.drop(columns=[self.target_col], inplace=True)
        
        # 2. трансформация, кодирование
        if self.categorical_cols:
            encoded_array = self.encoder.transform(X_transformed[self.categorical_cols])
            feature_names = self.encoder.get_feature_names_out(self.categorical_cols)
            
            # типа файнал дф
            encoded_df = pd.DataFrame(
                encoded_array, 
                columns=feature_names, 
                index=X_transformed.index
            )
            
            # дропаем старые и добавляем новые колонки
            X_transformed.drop(columns=self.categorical_cols, inplace=True)
            X_transformed = pd.concat([X_transformed, encoded_df], axis=1)
            
        return X_transformed, y

### класс для разделения данных для модели

In [23]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state

    def split(self, X, y):
        if y is None:
            raise ValueError("Целевая переменная y равна None. Проверьте названия колонок в FeatureExtractor и MyOneHotEncoder.")
        
        # вытаскиваем трейн и тест
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, 
            test_size=self.test_size, 
            random_state=self.random_state, 
            stratify=y
        )
        
        # из тренировочных данных вытаскиваем валидные
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_full, y_train_full, 
            test_size=self.test_size, 
            random_state=self.random_state, 
            stratify=y_train_full
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [40]:
class ModelSelection:
    # инит для словаря с grids и названиями моделей
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results_df = pd.DataFrame()
        self.best_model_name = None
    
    # обучение, лучшие параметры, лучший классификатор на основе валидационной выборке
    def choose(self, X_train, y_train, X_valid, y_valid):
        results_list = []
        best_valid_score = -1.0

        results_list = []
        best_valid_score = -1.0

        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict.get(idx, f"Model {idx}")
            print(f"Estimator: {model_name}")

            grid.fit(X_train, y_train)

            best_params = grid.best_params_
            best_train_score = grid.best_score_

            best_estimator = grid.best_estimator_
            y_pred = best_estimator.predict(X_valid)
            valid_score = accuracy_score(y_valid, y_pred)

            results_list.append({
                'model': model_name,
                'params': best_params,
                'valid_score': valid_score
            })

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")

            if valid_score > best_valid_score:
                best_valid_score = valid_score
                self.best_model_name = model_name

        print(f"Classifier with best validation set accuracy: {self.best_model_name}\n")

        self.results_df = pd.DataFrame(results_list)
        
        return self.best_model_name

    # ретерним дф с резами лучшей модели ич класса
    def best_results(self):
        return self.results_df

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [12]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    # обучаем на тренировочных и проверяем на тестрвых
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)

        y_pred = self.estimator.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        print(f"Accuracy of the final model is {acc}")
        
        return acc
    
    # сейв модел
    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [24]:
df = pd.read_csv('../data/checker_submits(1).csv')

### пайплан - предобработка данных к пригодному виду для компьютера

In [25]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

In [26]:
X, y = preprocessing.fit_transform(df)

In [27]:
splitter = TrainValidationTest(test_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)

### фьючи для перебора

In [28]:
svm_params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}

dt_params = {
    'max_depth': [None, 5, 10, 20, 30],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None],
    'random_state': [21]
}

rf_params = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 5, 10, 20, 30],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None],
    'random_state': [21]
}

### грид серч для каждой модели

In [29]:
gs_svm = GridSearchCV(SVC(), svm_params, scoring='accuracy', cv=3, n_jobs=-1)
gs_dt = GridSearchCV(DecisionTreeClassifier(), dt_params, scoring='accuracy', cv=3, n_jobs=-1)
gs_rf = GridSearchCV(RandomForestClassifier(), rf_params, scoring='accuracy', cv=3, n_jobs=-1)

In [30]:
grids = [gs_svm, gs_dt, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

In [41]:
selector = ModelSelection(grids, grid_dict)
best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)

Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.878
Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': None, 'random_state': 21}
Best training accuracy: 0.848
Validation set accuracy score for best params: 0.863
Estimator: Random Forest
Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 0.876
Validation set accuracy score for best params: 0.896
Classifier with best validation set accuracy: Random Forest



In [32]:
print("\nBest res: ")
print(selector.best_results())


Best res: 
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': 'balanced', 'criterion': 'gin...   
2  Random Forest  {'class_weight': 'balanced', 'criterion': 'ent...   

   valid_score  
0     0.877778  
1     0.862963  
2     0.896296  


In [33]:
best_grid_index = list(grid_dict.values()).index(best_model_name)
best_estimator = grids[best_grid_index].best_estimator_

In [34]:
finalizer = Finalize(best_estimator)

### для финального обучения юзаем контакт ту поссибилити ту юз макс дейта

In [35]:
X_train_full = pd.concat([X_train, X_valid])
y_train_full = pd.concat([y_train, y_valid])

In [36]:
accuracy = finalizer.final_score(X_train_full, y_train_full, X_test, y_test)

Accuracy of the final model is 0.9201183431952663


In [39]:
filename = f"{best_model_name}_{accuracy}.sav"
finalizer.save_model(filename)

Model was successfully saved to Random Forest_0.9201183431952663.sav
